# Sliding Window Contextual Embeddings

Generates XLM-RoBERTa contextual embeddings for all three languages (EN, HE, AR)
using a **16-word sliding window** centered on each target word.

## How it works

For each target word at position i in the transcript:
1. Take words at positions max(0, i-8) to min(N, i+8) — 16 words of context
2. Join them into a pseudo-sentence
3. Run XLM-RoBERTa on the pseudo-sentence
4. Extract the vector for the target word at its position in the window
5. If the target is a multi-word phrase (e.g. 'few_years'), average the component vectors

## Input
- `translated_podcast_transcript_filtered.csv` — 1735 words with en/he/ar columns
  (already in time order — this IS the transcript)

## Output
- `en_sliding_window_embeddings.csv` — (1735, 768)
- `he_sliding_window_embeddings.csv` — (1735, 768)
- `ar_sliding_window_embeddings.csv` — (1735, 768)

## 1. Load Data

In [ ]:
import pandas as pd
import torch
import numpy as np
import unicodedata
from pathlib import Path
from transformers import AutoTokenizer, AutoModel

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / 'data').exists() and (path / 'notebooks').exists():
            return path
    raise FileNotFoundError('Could not find project root containing data/ and notebooks/')

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'

TRANSCRIPT_PATH = DATA_DIR / 'sentences' / 'translated_podcast_transcript_filtered.csv'
if not TRANSCRIPT_PATH.exists():
    raise FileNotFoundError(f'{TRANSCRIPT_PATH} not found')

print('Loading data...')
word_level_df = pd.read_csv(TRANSCRIPT_PATH)
print(f'Project root: {PROJECT_ROOT}')
print(f'Transcript  : {TRANSCRIPT_PATH}')

print(f'Total words: {len(word_level_df)}')
print(f'Columns    : {word_level_df.columns.tolist()}')
print(f'\nFirst 5 rows:')
print(word_level_df[['start', 'en', 'he', 'ar']].head())

## 2. Configuration

In [ ]:
# Window size: number of words on each side of the target word
# Total context = 2 * HALF_WINDOW + 1 words (the target itself)
HALF_WINDOW = 8

# Languages to process
LANGUAGES = ['en', 'he', 'ar']

# Output paths
OUTPUT_PATHS = {
    'en': PROCESSED_DIR / 'en_sliding_window_embeddings.csv',
    'he': PROCESSED_DIR / 'he_sliding_window_embeddings.csv',
    'ar': PROCESSED_DIR / 'ar_sliding_window_embeddings.csv',
}

print(f'Window size  : {2 * HALF_WINDOW} words total ({HALF_WINDOW} each side)')
print(f'Languages    : {LANGUAGES}')

## 3. Helpers

In [ ]:
def normalize_text(text, lang):
    """
    Language-specific normalization for matching only.
    The window text fed to RoBERTa is always the raw original.
    """
    text = unicodedata.normalize('NFC', str(text))
    
    if lang == 'he':
        # Strip Hebrew niqqud (U+05B0-U+05C7)
        text = ''.join(c for c in text if not ('ְ' <= c <= 'ׇ'))
        text = text.replace('׳', '').replace('״', '')
    
    elif lang == 'ar':
        # Strip Arabic tashkeel (U+064B-U+065F) and tatweel (U+0640)
        text = ''.join(c for c in text if not ('ً' <= c <= 'ٟ'))
        text = text.replace('ـ', '')
    
    # Strip apostrophes and edge punctuation
    text = text.replace('’', '').replace("'", '').replace('`', '')
    text = text.strip(' .,!?"()-:;[]{}،؟؛־')
    return text.lower()


print('Helpers ready.')

## 4. Load Model

In [ ]:
print('Loading XLM-RoBERTa...')
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
model = AutoModel.from_pretrained('xlm-roberta-base')
model.eval()
print('Model ready.')

## 5. Generate Embeddings for All Three Languages

In [ ]:
results = {}  # lang -> {'embeddings': list, 'exact_matches': int, 'fallbacks': int}

def build_window_from_rows(word_list_raw, target_idx, half_window, lang):
    """
    Build a window of words centered on target_idx using row indices directly.
    Each row is one semantic unit (single word or phrase).
    No flattening — the window represents 16 content words from Eyal's word list.
    """
    start = max(0, target_idx - half_window)
    end   = min(len(word_list_raw), target_idx + half_window + 1)
    
    window_rows = word_list_raw[start:end]
    target_pos  = target_idx - start  # position of target within window
    
    # Convert each row to a display string for RoBERTa
    # English: few_years -> "few years" (underscore to space)
    # Hebrew/Arabic: keep as-is (spaces already present in multi-word entries)
    if lang == 'en':
        window_words = [str(w).strip().replace('_', ' ') for w in window_rows]
    else:
        window_words = [str(w).strip() for w in window_rows]
    
    return window_words, target_pos


for lang in LANGUAGES:
    print(f'\n{"="*50}')
    print(f'Processing {lang.upper()}...')
    print(f'{"="*50}')
    
    # Get raw word list for this language
    word_list_raw = word_level_df[lang].tolist()
    
    # Pre-compute normalized target components for each word
    # English: split on underscore -> ['few', 'years']
    # Hebrew/Arabic: split on space -> ['בני', 'אדם']
    split_char = '_' if lang == 'en' else ' '
    
    target_components_list = []
    for raw in word_list_raw:
        parts = str(raw).strip().split(split_char)
        components = [normalize_text(p, lang) for p in parts if normalize_text(p, lang)]
        target_components_list.append(components)
    
    print(f'  Word list entries : {len(word_list_raw)}')
    
    # Extract embeddings
    embeddings = []
    exact_matches = 0
    fallbacks = 0
    
    for wi in range(len(word_level_df)):
        components = target_components_list[wi]
        phrase_len = len(components)
        
        # Build window from row indices — no flattening
        window_words, target_pos = build_window_from_rows(
            word_list_raw, wi, HALF_WINDOW, lang
        )
        
        # Join window into pseudo-sentence for RoBERTa
        # Each entry is a word or phrase, joined by spaces
        pseudo_sentence = ' '.join(window_words)
        
        # Tokenize and run model
        encoded = tokenizer(
            pseudo_sentence,
            return_tensors='pt',
            truncation=True,
            max_length=512
        )
        
        with torch.no_grad():
            outputs = model(**encoded)
        
        token_embeddings = outputs.last_hidden_state.squeeze(0)
        word_ids = encoded.word_ids()
        input_ids = encoded['input_ids'][0]
        
        # Group subtoken vectors by whitespace-split word index
        word_vectors = {}
        word_token_ids = {}
        for idx, word_id in enumerate(word_ids):
            if word_id is None:
                continue
            if word_id not in word_vectors:
                word_vectors[word_id] = []
                word_token_ids[word_id] = []
            word_vectors[word_id].append(token_embeddings[idx].numpy())
            word_token_ids[word_id].append(input_ids[idx].item())
        
        # Build normalized token list
        window_token_list = []
        for wid in sorted(word_vectors.keys()):
            avg_vec   = np.mean(word_vectors[wid], axis=0)
            raw_text  = tokenizer.decode(word_token_ids[wid])
            norm_text = normalize_text(raw_text, lang)
            if norm_text:
                window_token_list.append((norm_text, avg_vec))
        
        # Find target word in token list
        # Search around expected position first, then expand
        matched_at = None
        search_order = list(range(
            max(0, target_pos - 2),
            min(len(window_token_list) - phrase_len + 1, target_pos + 3)
        ))
        search_order += [i for i in range(len(window_token_list) - phrase_len + 1)
                         if i not in search_order]
        
        for start_pos in search_order:
            if start_pos + phrase_len > len(window_token_list):
                continue
            if all(
                window_token_list[start_pos + j][0] == components[j]
                for j in range(phrase_len)
            ):
                matched_at = start_pos
                break
        
        if matched_at is not None:
            # Exact match — average component vectors
            component_vecs = [
                window_token_list[matched_at + j][1]
                for j in range(phrase_len)
            ]
            vec = np.mean(component_vecs, axis=0)
            exact_matches += 1
        else:
            # Positional fallback — use vector at expected target position
            # This is still a genuine contextual vector from the correct window
            if target_pos < len(window_token_list):
                vec = window_token_list[target_pos][1]
            elif window_token_list:
                vec = np.mean([v for _, v in window_token_list], axis=0)
            else:
                vec = np.zeros(768)
            fallbacks += 1
        
        embeddings.append(vec)
        
        # Progress every 100 words
        if (wi + 1) % 100 == 0:
            print(f'  [{wi+1:4d}/{len(word_level_df)}] '
                  f'exact={exact_matches} fallback={fallbacks}')
    
    results[lang] = {
        'embeddings': embeddings,
        'exact_matches': exact_matches,
        'fallbacks': fallbacks
    }
    
    print(f'\n  Done {lang.upper()}:')
    print(f'  Exact word match : {exact_matches} / {len(word_level_df)} '
          f'({exact_matches/len(word_level_df)*100:.1f}%)')
    print(f'  Position fallback: {fallbacks} / {len(word_level_df)} '
          f'({fallbacks/len(word_level_df)*100:.1f}%)')

print('\nAll languages processed.')

## 6. Report

In [ ]:
print('=' * 55)
print('SLIDING WINDOW EMBEDDING REPORT')
print('=' * 55)
print(f'Window size: {2 * HALF_WINDOW} words ({HALF_WINDOW} each side)')
print(f'Total target words: {len(word_level_df)}')
print()

for lang in LANGUAGES:
    r = results[lang]
    embs = np.array(r['embeddings'])
    print(f'{lang.upper()}:')
    print(f'  Shape         : {embs.shape}')
    print(f'  Exact matches : {r["exact_matches"]} ({r["exact_matches"]/len(word_level_df)*100:.1f}%)')
    print(f'  Fallbacks     : {r["fallbacks"]} ({r["fallbacks"]/len(word_level_df)*100:.1f}%)')
    print(f'  Any NaN       : {np.isnan(embs).any()}')
    assert embs.shape == (len(word_level_df), 768), f'Shape error for {lang}'
    assert not np.isnan(embs).any(), f'NaN found in {lang}'
    print(f'  Verified ✓')
    print()

print('Note: "Exact match" = target word found at its expected position in the window.')
print('      "Fallback"    = target word not found by string match; used positional vector.')
print('      Both types are genuine contextual vectors — the window provides real context.')

## 7. Save

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

for lang in LANGUAGES:
    embs = np.array(results[lang]['embeddings'])
    out_path = OUTPUT_PATHS[lang]
    pd.DataFrame(embs).to_csv(out_path, index=False)
    print(f'Saved {lang.upper()} -> {out_path.resolve()}  shape: {embs.shape}')

print('\nAll embeddings saved.')
print('Next step: run 05_Projection_Residuals.ipynb using these files.')